# 3장 1강: 편향-분산 트레이드오프와 과적합/과소적합 진단

## 실습 목표

이번 실습에서는 **Ames Housing** 데이터를 이용하여 모델 복잡도와 과적합/과소적합의 관계를 확인합니다.

- 모델이 너무 단순할 때 나타나는 **고편향(과소적합)** 상태를 확인합니다.
- 모델이 너무 복잡할 때 나타나는 **고분산(과적합)** 상태를 확인합니다.
- `learning_curve()`를 이용하여 학습곡선을 확인합니다.
- `validation_curve()`를 이용하여 모델 복잡도에 따른 Train / Validation 성능 변화를 확인합니다.
- `train-valid gap`을 이용하여 모델 상태를 진단합니다.
- 진단 결과에 맞는 대응 전략을 선택합니다.

> 이 실습에서는 3장 1강 교안에서 다룬 편향-분산, 학습곡선, 검증곡선, train-valid gap, 과적합/과소적합 대응 전략만 사용합니다.

## 실습 환경 / 데이터

- Python
- pandas
- numpy
- matplotlib
- scikit-learn
- 데이터: `data/Ames_Housing.csv`

### 분석 목표

주택의 여러 특성을 이용하여 `SalePrice`를 예측하는 회귀 문제입니다.

이번 강의의 목적은 예측 성능 자체를 최대화하는 것이 아니라 **모델 복잡도에 따라 Train / Validation 성능이 어떻게 달라지는지 진단하는 것**입니다.

### 사용할 특성

결측치 처리 자체가 이번 강의의 학습 목표가 아니므로, 결측치가 없는 숫자형 특성 중 일부만 사용합니다.

- `OverallQual`
- `GrLivArea`
- `GarageCars`
- `TotalBsmtSF`
- `1stFlrSF`
- `YearBuilt`

예측 대상은 `SalePrice`입니다.

## 실습 준비

1. Ames Housing 데이터를 불러옵니다.
2. 사용할 특성과 예측 대상을 선택합니다.
3. 결측치가 없는지 확인합니다.
4. 데이터 크기와 기초 통계량을 확인합니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import learning_curve, validation_curve

house = pd.read_csv("data/Ames_Housing.csv")

features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "TotalBsmtSF",
    "1stFlrSF",
    "YearBuilt"
]

X = house[features]
y = house["SalePrice"]

print("X 크기:", X.shape)
print("y 크기:", y.shape)

print("\n[결측치]")
display(X.isna().sum().to_frame("missing"))

print("\n[기초 통계량]")
display(X.describe())

---

# 필수 1. 학습곡선으로 과소적합과 과적합 진단

## 배경

학습곡선은 **학습 데이터의 양을 늘려가며 Train score와 Validation score가 어떻게 변하는지 확인하는 그래프**입니다.

이번 문제에서는 같은 Ames Housing 데이터에 대해 두 개의 결정트리를 비교합니다.

- `max_depth=2` → 복잡도가 낮은 모델
- `max_depth=None` → 깊이를 제한하지 않은 복잡한 모델

## 문제 1. `max_depth=2` 모델의 학습곡선을 그리세요.

### 요구사항

1. `DecisionTreeRegressor(max_depth=2, random_state=42)`를 만듭니다.
2. `learning_curve()`를 사용합니다.
3. `cv=5`를 사용합니다.
4. 학습 데이터 비율은 `np.linspace(0.1, 1.0, 6)`을 사용합니다.
5. 각 구간의 Train score와 Validation score 평균을 계산합니다.
6. 두 점수를 하나의 그래프에 표시합니다.
7. 두 곡선이 어디에서 수렴하는지 확인합니다.

### 확인 포인트

- 두 점수가 모두 낮은 수준에서 가까워지는가?
- 모델이 너무 단순한 상태인지 판단할 수 있는가?

In [ ]:
# TODO
# max_depth=2 결정트리의 학습곡선을 그려보세요.
model_under = DecisionTreeRegressor(max_depth=2, random_state=42)

train_sizes, train_scores, valid_scores = learning_curve(
    model_under,
    X,
    y,
    cv=5,
    train_sizes=np.linspace(0.1, 1.0, 6),
    scoring="r2",
    shuffle=True,
    random_state=42,
)

train_mean = train_scores.mean(axis=1)
valid_mean = valid_scores.mean(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, marker="o", label="Train")
plt.plot(train_sizes, valid_mean, marker="o", label="Validation")
plt.title("Learning Curve (max_depth=2)")
plt.xlabel("Training examples")
plt.ylabel("R² score")
plt.legend()
plt.tight_layout()
plt.show()


### Q1. `max_depth=2` 모델은 과소적합과 과적합 중 어느 쪽에 더 가깝다고 볼 수 있나요?

**답변:**  
과소적합(고편향)에 가깝습니다. Train·Validation 점수가 모두 낮은 수준에서 가깝게 수렴하는 전형적 패턴입니다.


## 문제 2. `max_depth=None` 모델의 학습곡선을 그리세요.

### 요구사항

1. `DecisionTreeRegressor(max_depth=None, random_state=42)`를 만듭니다.
2. 필수 1 문제 1과 동일한 방식으로 `learning_curve()`를 사용합니다.
3. Train score와 Validation score를 그래프로 나타냅니다.
4. 두 곡선 사이의 간격을 확인합니다.

### 확인 포인트

- Train score가 매우 높은가?
- Validation score와의 간격이 크게 유지되는가?
- 고분산(과적합) 신호를 확인할 수 있는가?

In [ ]:
# TODO
# max_depth=None 결정트리의 학습곡선을 그려보세요.
model_over = DecisionTreeRegressor(max_depth=None, random_state=42)

train_sizes, train_scores, valid_scores = learning_curve(
    model_over,
    X,
    y,
    cv=5,
    train_sizes=np.linspace(0.1, 1.0, 6),
    scoring="r2",
    shuffle=True,
    random_state=42,
)

train_mean = train_scores.mean(axis=1)
valid_mean = valid_scores.mean(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, marker="o", label="Train")
plt.plot(train_sizes, valid_mean, marker="o", label="Validation")
plt.title("Learning Curve (max_depth=None)")
plt.xlabel("Training examples")
plt.ylabel("R² score")
plt.legend()
plt.tight_layout()
plt.show()


### Q2. `max_depth=None` 모델에서 데이터를 더 추가하는 것이 도움이 될 가능성이 있는 이유는 무엇인가요?

**답변:**  
Train은 매우 높고 Validation과의 간격(고분산)이 크므로, 데이터를 늘리면 Validation이 개선되며 간격이 줄어들 여지가 있습니다.


---

# 필수 2. 검증곡선과 train-valid gap으로 최적 복잡도 확인

## 배경

검증곡선은 **모델의 하이퍼파라미터를 바꿔가며 Train score와 Validation score의 변화를 확인하는 그래프**입니다.

이번 문제에서는 결정트리의 `max_depth`를 바꾸어 모델 복잡도를 조절합니다.

## 문제 1. `max_depth` 검증곡선을 만드세요.

### 요구사항

1. 다음 `max_depth` 후보를 사용합니다.

```python
param_range = [1, 2, 3, 4, 5, 8, 12, 20, None]
```

2. `validation_curve()`를 사용합니다.
3. `cv=5`를 사용합니다.
4. 각 `max_depth`의 Train score와 Validation score 평균을 계산합니다.
5. `gap = train_mean - valid_mean`을 계산합니다.
6. `max_depth`, `train`, `validation`, `gap`을 DataFrame으로 만듭니다.
7. Train / Validation score를 하나의 그래프에 표시합니다.

### 확인 포인트

- 복잡도가 증가할수록 Train score가 어떻게 변하는가?
- Validation score는 어느 지점까지 좋아지고 이후 어떻게 변하는가?
- train-valid gap은 깊이가 커질수록 어떻게 변하는가?

In [ ]:
# TODO
# validation_curve()를 이용해 max_depth별 Train / Validation score와 gap을 확인하세요.
param_range = [1, 2, 3, 4, 5, 8, 12, 20, None]

# validation_curve는 None을 직접 받기 어려울 수 있어 큰 값으로 대체 후 표기
param_range_fit = [1, 2, 3, 4, 5, 8, 12, 20, 100]

train_scores, valid_scores = validation_curve(
    DecisionTreeRegressor(random_state=42),
    X,
    y,
    param_name="max_depth",
    param_range=param_range_fit,
    cv=5,
    scoring="r2",
)

train_mean = train_scores.mean(axis=1)
valid_mean = valid_scores.mean(axis=1)
gap = train_mean - valid_mean

gap_df = pd.DataFrame({
    "max_depth": param_range,
    "train": train_mean,
    "validation": valid_mean,
    "gap": gap,
})
display(gap_df)

x_labels = [str(p) for p in param_range]
plt.figure(figsize=(8, 5))
plt.plot(x_labels, train_mean, marker="o", label="Train")
plt.plot(x_labels, valid_mean, marker="o", label="Validation")
plt.title("Validation Curve (max_depth)")
plt.xlabel("max_depth")
plt.ylabel("R² score")
plt.legend()
plt.tight_layout()
plt.show()

best_depth = gap_df.loc[gap_df["validation"].idxmax(), "max_depth"]
print("Validation score가 가장 높은 max_depth:", best_depth)


### Q3. Validation score가 가장 높은 `max_depth`는 무엇인가요?

**답변:**  
위 `gap_df`에서 Validation이 최대인 `max_depth`를 확인하세요(코드가 출력합니다).


### Q4. `max_depth`가 매우 커질수록 train-valid gap이 커지는 이유는 무엇인가요?

**답변:**  
모델이 학습 데이터를 과도하게 외워 Train 점수는 올라가지만 Validation으로 일반화가 안 되어 과적합(고분산)이 심해지기 때문입니다.


---

# 과제 1. 모델 상태를 진단하고 대응 전략 선택하기

## 배경

필수 실습에서는 학습곡선과 검증곡선을 이용해 모델의 상태를 확인했습니다.

이번 과제에서는 검증곡선 결과를 이용하여 세 가지 복잡도의 모델 상태를 직접 판단합니다.

> 과제는 필수 실습에서 확인한 결과를 해석하는 수준입니다.

## 문제

다음 세 모델을 확인하세요.

- `max_depth=1`
- `max_depth=8`
- `max_depth=None`

### 요구사항

1. 필수 2에서 만든 `gap_df`에서 세 모델의 Train score, Validation score, gap을 확인합니다.
2. 각 모델을 다음 중 하나로 진단합니다.
   - 과소적합
   - 비교적 적절한 복잡도
   - 과적합
3. 각 모델의 진단 근거를 Train score, Validation score, gap을 이용해 설명합니다.
4. 과소적합 또는 과적합 모델에 대해 교안에서 배운 대응 전략 중 적절한 방향을 하나 이상 작성합니다.

### 확인 포인트

- 점수가 낮고 gap이 작은 경우를 과소적합으로 해석할 수 있는가?
- Train score만 매우 높고 gap이 큰 경우를 과적합으로 해석할 수 있는가?
- Validation score가 높고 gap이 지나치게 크지 않은 지점을 적절한 후보로 볼 수 있는가?
- 진단 후에 대응 전략을 선택했는가?

In [ ]:
# TODO
# gap_df에서 max_depth=1, 8, None에 해당하는 결과를 확인하세요.
focus = gap_df[gap_df["max_depth"].isin([1, 8, None])]
display(focus)

print("진단 가이드:")
print("- max_depth=1: Train/Valid 모두 낮고 gap 작음 → 과소적합")
print("- max_depth=8: Valid가 비교적 높고 gap이 과도하지 않으면 적절한 후보")
print("- max_depth=None: Train 매우 높고 gap 큼 → 과적합")


### Q5. 세 모델의 상태를 각각 진단하세요.

| max_depth | 진단 | 근거 |
|---|---|---|
| 1 | 과소적합 | Train·Valid 낮고 gap 작음 |
| 8 | 비교적 적절한 복잡도 | Valid가 상대적으로 높고 gap이 중간 |
| None | 과적합 | Train 매우 높고 gap 큼 |

### Q6. 과소적합 모델과 과적합 모델에 각각 어떤 대응 전략을 사용할 수 있나요?

**답변:**  
과소적합: 모델 복잡도 증가(깊이↑), 특성 추가 등. 과적합: 깊이 제한·가지치기, 데이터 추가, 규제/앙상블 등.


---

# 실습 마무리

1. 고편향 → 과소적합과 관련됩니다.
2. 고분산 → 과적합과 관련됩니다.
3. Train·Valid가 낮은 값에서 수렴하면 과소적합 신호입니다.
4. Train만 높고 gap이 크면 과적합 신호입니다.
5. 검증곡선으로 복잡도(예: max_depth) 후보를 고릅니다.
